# Tahap 3: Retrieval Model

Vektorisasi TF-IDF dan inisiasi model pencarian kosinus.

*Notebook ini dihasilkan secara otomatis dari skrip `notebooks/case3.py` dan telah dieksekusi penuh.*

In [1]:
import os
# Menyesuaikan working directory ke root proyek
os.chdir('..')
print('Current Working Directory:', os.getcwd())

Current Working Directory: E:\IQBAL\TUGAS KULIAH\SEMESTER 6\PENALARAN KOMPUTER\SubCPMK4 Genap 2025-2026\cbr_merek


### -*- coding: utf-8 -*-

In [2]:
# -*- coding: utf-8 -*-
"""
Tugas Penalaran Komputer - SIKLUS CBR (Tahap 3: Case Retrieval)
Studi Kasus: Sengketa Merek & Indikasi Geografis (UU No. 20 Tahun 2016)
Fakultas Teknik - Informatika UMM
"""

import os
import re
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.svm import SVC

### KONFIGURASI DAN PENYEDIAAN DIREKTORI

In [3]:
# KONFIGURASI DAN PENYEDIAAN DIREKTORI

In [4]:
PROCESSED_JSON_PATH = "data/processed/cases.json"
EVAL_DIR = "data/eval"
QUERIES_JSON_PATH = os.path.join(EVAL_DIR, "queries.json")

os.makedirs(EVAL_DIR, exist_ok=True)

### FUNGSI RAPI PENYEDIAAN TEKS (PREPROCESSING)

In [5]:
# FUNGSI RAPI PENYEDIAAN TEKS (PREPROCESSING)

In [6]:
def preprocess_query(text):
    """
    Membersihkan teks pertanyaan/query sebelum ditukar menjadi representasi vektor.
    """
    text = text.lower()
    text = re.sub(r'[^\w\s\-\/\.]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

### 1. MEMBACA PANGKALAN DATA KES (LOAD CASE BASE)

In [7]:
# 1. MEMBACA PANGKALAN DATA KES (LOAD CASE BASE)

In [8]:
if not os.path.exists(PROCESSED_JSON_PATH):
    print("[RALAT] Fail data terstruktur 'cases.json' tidak ditemui!")
    print("[INFO] Sila jalankan skrip Tahap 2 (02_case_representation.py) terlebih dahulu.")
    exit()

with open(PROCESSED_JSON_PATH, "r", encoding="utf-8") as f:
    cases_db = json.load(f)

print(f"[INFO] Berjaya memuatkan {len(cases_db)} kes dari pangkalan data.")

[INFO] Berjaya memuatkan 69 kes dari pangkalan data.


### 2. PEMBAHAGIAN DATA (SPLITTING DATA)

In [9]:
# 2. PEMBAHAGIAN DATA (SPLITTING DATA)

### Kita membahagikan data kepada data latihan (Train) dan data ujian (Test)

In [10]:
# Kita membahagikan data kepada data latihan (Train) dan data ujian (Test)
# dengan nisbah standard akademis 80:20 mengikut arahan modul tugasan.
train_cases, test_cases = train_test_split(cases_db, test_size=0.2, random_state=42)
print(f"[INFO] Pembahagian Data Selesai (Nisbah 80:20):")
print(f"       └─ Data Latihan (Train Set): {len(train_cases)} kes")
print(f"       └─ Data Ujian (Test Set)   : {len(test_cases)} kes")

[INFO] Pembahagian Data Selesai (Nisbah 80:20):
       └─ Data Latihan (Train Set): 55 kes
       └─ Data Ujian (Test Set)   : 14 kes


### 3. REPRESENTASI VEKTOR (TF-IDF VECTORIZATION)

In [11]:
# 3. REPRESENTASI VEKTOR (TF-IDF VECTORIZATION)

### Membina pembina vektor TF-IDF menggunakan teks penuh yang telah dibersihkan

In [12]:
# Membina pembina vektor TF-IDF menggunakan teks penuh yang telah dibersihkan
train_texts = [case["text_full"] for case in train_cases]
vectorizer = TfidfVectorizer(preprocessor=preprocess_query)
tfidf_train_matrix = vectorizer.fit_transform(train_texts)

# Sediakan juga matriks TF-IDF untuk keseluruhan database bagi tujuan carian penuh
all_texts = [case["text_full"] for case in cases_db]
tfidf_all_matrix = vectorizer.transform(all_texts)

print(f"[INFO] Selesai membina vektor TF-IDF. Jumlah dimensi kosa kata: {len(vectorizer.vocabulary_)} kata.")

[INFO] Selesai membina vektor TF-IDF. Jumlah dimensi kosa kata: 5269 kata.


### 4. MODEL RETRIEVAL / KLASIFIKASI (SVM MODEL)

In [13]:
# 4. MODEL RETRIEVAL / KLASIFIKASI (SVM MODEL)

### Melatih model Machine Learning Support Vector Machine (SVM) pada representasi TF-IDF

In [14]:
# Melatih model Machine Learning Support Vector Machine (SVM) pada representasi TF-IDF
# untuk mengkelaskan keputusan sengketa (solusi_hukum).
train_labels = [case["solusi_hukum"] for case in train_cases]

# Kami telah membuang parameter probability=True untuk menghilangkan FutureWarning
# pada pustaka Scikit-Learn terbaru (1.9+) karena kita hanya memerlukan fungsi prediksi kelas langsung.
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(tfidf_train_matrix, train_labels)

print("[INFO] Model Klasifikasi SVM berjaya dilatih pada representasi TF-IDF.")

[INFO] Model Klasifikasi SVM berjaya dilatih pada representasi TF-IDF.


### 5. FUNGSI RETRIEVAL UTAMA (RETRIEVE FUNCTION)

In [15]:
# 5. FUNGSI RETRIEVAL UTAMA (RETRIEVE FUNCTION)

In [16]:
def retrieve(query: str, k: int = 5):
    """
    Mencari k-kes terdahulu yang paling serupa dengan kes baru (query).
    
    Langkah Kerja:
    1) Pre-process query
    2) Hitung vektor query
    3) Hitung cosine-similarity dengan semua case vectors
    4) Kembalikan top-k case_id beserta skor kemiripan
    """
    # 1) Pre-process query
    cleaned_query = preprocess_query(query)
    
    # 2) Hitung vektor query
    query_vector = vectorizer.transform([cleaned_query])
    
    # 3) Hitung cosine-similarity dengan semua case vectors dalam pangkalan data
    similarities = cosine_similarity(query_vector, tfidf_all_matrix).flatten()
    
    # 4) Dapatkan indeks top-k kes teratas mengikut kemiripan tertinggi
    top_k_indices = np.argsort(similarities)[::-1][:k]
    
    retrieved_results = []
    for idx in top_k_indices:
        case = cases_db[idx]
        retrieved_results.append({
            "case_id": case["case_id"],
            "no_perkara": case["no_perkara"],
            "pihak": case["pihak"],
            "merek_penggugat": case["merek_penggugat"],
            "merek_tergugat": case["merek_tergugat"],
            "similarity": float(similarities[idx]),
            "solusi_hukum": case["solusi_hukum"],
            "ringkasan_fakta": case["ringkasan_fakta"]
        })
        
    return retrieved_results

### 6. PENYEDIAAN KES UJIAN SECARA DINAMIK (DYNAMICAL TEST QUERY CREATOR)

In [17]:
# 6. PENYEDIAAN KES UJIAN SECARA DINAMIK (DYNAMICAL TEST QUERY CREATOR)

### Untuk mengelakkan ralat 'hardcoded case_id' yang tidak sepadan dengan data asli anda,

In [18]:
# Untuk mengelakkan ralat 'hardcoded case_id' yang tidak sepadan dengan data asli anda,
# kami membina penjana query automatik yang mengambil fakta dari pangkalan data sebenar
# dan menetapkannya sebagai Ground-Truth secara dinamik!

def load_test_queries():
    """
    Memuat kes ujian pengesahan (queries.json) yang telah dibuat secara independen.
    """
    print(f"[INFO] Memuat kes ujian pengesahan dari '{QUERIES_JSON_PATH}'...")
    with open(QUERIES_JSON_PATH, "r", encoding="utf-8") as json_in:
        test_queries_list = json.load(json_in)
    print(f"[✔] Berjaya memuat {len(test_queries_list)} kes ujian.")
    return test_queries_list

test_queries = load_test_queries()

[INFO] Memuat kes ujian pengesahan dari 'data/eval\queries.json'...
[✔] Berjaya memuat 7 kes ujian.


### 7. PENGUJIAN AWAL PIPELINE RETRIEVAL

In [19]:
# 7. PENGUJIAN AWAL PIPELINE RETRIEVAL

In [20]:
def run_initial_retrieval_test():
    print("\n" + "="*80)
    # Menampilkan tajuk ujian menggunakan bahasa Indonesia/Melayu formal
    print(" DEMO PENGUJIAN AWAL RETRIEVAL (TF-IDF & COSINE SIMILARITY)")
    print("="*80)
    
    for q in test_queries[:3]:  # Papar 3 query sahaja untuk demo konsol yang bersih
        print(f"\n[Query Ujian] ID: {q['query_id']}")
        print(f"  └─ Fakta Baru : \"{q['query_text'][:120]}...\"")
        print(f"  └─ Sasaran GT : {q['ground_truth_case_id']} ({q['ground_truth_solusi']})")
        
        # Jalankan fungsi retrieve() utama
        results = retrieve(q["query_text"], k=3)
        
        # Gunakan model SVM untuk meramal kelas sengketa secara langsung
        query_vector = vectorizer.transform([preprocess_query(q["query_text"])])
        predicted_class = svm_model.predict(query_vector)[0]
        
        print(f"  └─ Hasil Carian CBR (Top-3 Kes Serupa):")
        for rank, res in enumerate(results):
            indicator = "⭐ [TEPAT]" if res["case_id"] == q["ground_truth_case_id"] else " "
            print(f"     {rank+1}. Kes ID: {res['case_id']} | Nilai Mirip: {res['similarity']:.4f} | Merek: {res['merek_penggugat']} VS {res['merek_tergugat']} {indicator}")
            
        print(f"  └─ Ramalan Keputusan Model (SVM): {predicted_class}")
        print("-" * 60)

if __name__ == "__main__":
    run_initial_retrieval_test()


 DEMO PENGUJIAN AWAL RETRIEVAL (TF-IDF & COSINE SIMILARITY)

[Query Ujian] ID: Q001
  └─ Fakta Baru : "Sebuah perusahaan makanan cepat saji asal Filipina yang telah beroperasi di Indonesia sejak lama mendapati pihak lain me..."
  └─ Sasaran GT : case_01 (GUGATAN DIKABULKAN (PEMBATALAN / PENGHAPUSAN MEREK TERGUGAT))
  └─ Hasil Carian CBR (Top-3 Kes Serupa):
     1. Kes ID: case_01 | Nilai Mirip: 0.2255 | Merek: JOLLIBEE VS JOLLIBE E ⭐ [TEPAT]
     2. Kes ID: case_45 | Nilai Mirip: 0.2085 | Merek: GOLDEN VALLEY VS GO LDEN VALLEY  
     3. Kes ID: case_23 | Nilai Mirip: 0.1928 | Merek: FARMSTAY VS FARMS TAY  
  └─ Ramalan Keputusan Model (SVM): GUGATAN DITOLAK
------------------------------------------------------------

[Query Ujian] ID: Q002
  └─ Fakta Baru : "Penggugat adalah pemilik merek dagang produk pelumas mesin yang telah terdaftar di kelas yang relevan. Pihak tergugat me..."
  └─ Sasaran GT : case_02 (GUGATAN DITOLAK)
  └─ Hasil Carian CBR (Top-3 Kes Serupa):
     1. Kes ID: ca